In [1]:
import pandas as pd

df = pd.read_parquet("predictions_checkpoint.parquet")
df.head()

,gml_id,volume_m3,label_en,osm_building_type,osm_landuse_class,osm_landuse_name,gfk_class,ALKIS_Landuse_info,osm_names,alkis_address,...,shop,tourism,information,tags_search,additional_information,website,sentence,interpreted_type,mid_labels,bosserhof_class
0,8,35.568276,Buildings for business or commerce,None,residential,None,Gebäude,residence,None,"Braunschweig, Stadt",...,None,None,None,None,None,None,This row describes one building. Building iden...,error,[],None
1,9,101.096224,Buildings for business or commerce,None,residential,None,Gebäude,residence,None,"Braunschweig, Stadt",...,None,None,None,None,None,None,This row describes one building. Building iden...,error,[],None
2,10,1316.648979,residential buildings,None,residential,None,Gebäude,residence,None,"Schaftrift 13 Braunschweig, Stadt",...,None,None,None,None,None,None,This row describes one building. Building iden...,error,[],None
3,11,74.154358,Buildings for business or commerce,None,residential,None,Gebäude,residence,None,"Braunschweig, Stadt",...,None,None,None,None,None,None,This row describes one building. Building iden...,error,[],None
4,4,351.716079,Buildings for business or commerce,None,residential,None,Gebäude,residence,None,"Braunschweig, Stadt",...,None,None,None,None,None,None,This row describes one building. Building iden...,residential building,[],None


In [2]:
df['interpreted_type'].value_counts()

interpreted_type
residential building                                         189772
Residential building                                         105599
error                                                         52457
Commercial building                                           12597
commercial building                                            9243
                                                              ...  
Sympatec GmbH industrial facility                                 1
Hirschler Teich (pond/reservoir)                                  1
Bakery with residential units                                     1
Mixed-use building housing a dental clinic and apartments         1
DHL postal depot                                                  1
Name: count, Length: 36326, dtype: int64

In [3]:
df[df['interpreted_type'] == 'error'].to_parquet("error_predictions.parquet")

In [4]:
# Remove rows where interpreted_type == 'error'
df = df[df['interpreted_type'] != 'error']

In [5]:
df.to_parquet("clean_predictions.parquet")

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 548717 entries, 4 to 601173
Data columns (total 23 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   gml_id                  548717 non-null  object 
 1   volume_m3               548717 non-null  float64
 2   label_en                500148 non-null  object 
 3   osm_building_type       53354 non-null   object 
 4   osm_landuse_class       487885 non-null  object 
 5   osm_landuse_name        44425 non-null   object 
 6   gfk_class               500148 non-null  object 
 7   ALKIS_Landuse_info      475306 non-null  object 
 8   osm_names               16483 non-null   object 
 9   alkis_address           508782 non-null  object 
 10  email                   804 non-null     object 
 11  amenity                 6598 non-null    object 
 12  building                4423 non-null    object 
 13  shop                    3821 non-null    object 
 14  tourism                 1

In [7]:
df.head()

,gml_id,volume_m3,label_en,osm_building_type,osm_landuse_class,osm_landuse_name,gfk_class,ALKIS_Landuse_info,osm_names,alkis_address,...,shop,tourism,information,tags_search,additional_information,website,sentence,interpreted_type,mid_labels,bosserhof_class
4,4,351.716079,Buildings for business or commerce,None,residential,None,Gebäude,residence,None,"Braunschweig, Stadt",...,None,None,None,None,None,None,This row describes one building. Building iden...,residential building,[],None
5,5,91.824388,Buildings for business or commerce,None,residential,None,Gebäude,residence,None,"Braunschweig, Stadt",...,None,None,None,None,None,None,This row describes one building. Building iden...,residential building,[],None
6,6,9.680736,Buildings for business or commerce,None,residential,None,Gebäude,public_office,None,"Braunschweig, Stadt",...,None,None,None,None,None,None,This row describes one building. Building iden...,public office building,"[work, errands]",customer-oriented services
7,7,32.861208,Buildings for supply,None,residential,None,Gebäude,residence,None,"Braunschweig, Stadt",...,None,None,None,None,None,None,This row describes one building. Building iden...,residential building,[],None
8,12,51.354285,residential buildings,None,allotments,KGV Auf dem Klei,Gebäude,None,None,"Braunschweig, Stadt",...,None,None,None,None,None,None,This row describes one building. Building iden...,Allotment garden cottage (Kleingartenhaus),[leisure],None


In [8]:
import numpy as np

df['mid_label'] = df['mid_labels'].apply(
    lambda x: np.nan if (
        (isinstance(x, list) and len(x) == 0) or
        (isinstance(x, np.ndarray) and x.size == 0)
    ) else x
)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 548717 entries, 4 to 601173
Data columns (total 24 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   gml_id                  548717 non-null  object 
 1   volume_m3               548717 non-null  float64
 2   label_en                500148 non-null  object 
 3   osm_building_type       53354 non-null   object 
 4   osm_landuse_class       487885 non-null  object 
 5   osm_landuse_name        44425 non-null   object 
 6   gfk_class               500148 non-null  object 
 7   ALKIS_Landuse_info      475306 non-null  object 
 8   osm_names               16483 non-null   object 
 9   alkis_address           508782 non-null  object 
 10  email                   804 non-null     object 
 11  amenity                 6598 non-null    object 
 12  building                4423 non-null    object 
 13  shop                    3821 non-null    object 
 14  tourism                 1

In [10]:
df = df.reset_index()

In [12]:
import geopandas as gpd

gdf_geom = gpd.read_file("condensed_buildings_with_pois.gpkg")

gdf_geom["gml_id"] = gdf_geom["gml_id"].astype(str)
df["gml_id"] = df["gml_id"].astype(str)

geom_df = gdf_geom[["gml_id", "geometry"]].copy()

merged_df = df.merge(geom_df, on="gml_id", how="left")

final_gdf = gpd.GeoDataFrame(merged_df, geometry="geometry", crs=gdf_geom.crs)

final_gdf.to_file("final_predictions_with_geometry.gpkg", driver="GPKG")

In [13]:
print("Rows in parquet df:", len(df))
print("Rows after merge:", len(final_gdf))
print("Missing geometries:", final_gdf["geometry"].isna().sum())

Rows in parquet df: 548717
Rows after merge: 548717
Missing geometries: 0


In [14]:
final_gdf.columns

Index(['index', 'gml_id', 'volume_m3', 'label_en', 'osm_building_type',
       'osm_landuse_class', 'osm_landuse_name', 'gfk_class',
       'ALKIS_Landuse_info', 'osm_names', 'alkis_address', 'email', 'amenity',
       'building', 'shop', 'tourism', 'information', 'tags_search',
       'additional_information', 'website', 'sentence', 'interpreted_type',
       'mid_labels', 'bosserhof_class', 'mid_label', 'geometry'],
      dtype='object')